In [1]:
!pip install pytrec_eval gensim

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 53.8 MB/s eta 0:00:00
  Created wheel for pytrec_eval: filename=pytrec_eval-0.5-cp312-cp312-linux_x86_64.whl size=309344 sha256=336eca7267d785238a7778b45a64a32e42b14a86b1aeed2cc9a3ce6110d4ce02
  Stored in directory: /root/.cache/pip/wheels/c6/4a/9e/e17f9ea004e1c221bd0ff384732285211c4917b790d598ea51
Successfully built pytrec_eval


In [2]:
import json
import os
import pickle
import numpy as np
from collections import defaultdict
from numpy.linalg import norm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from gensim.models import KeyedVectors
from google.colab import drive
from scipy.sparse import save_npz, load_npz
import pytrec_eval
import subprocess

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ==================== CẤU HÌNH ====================
CORPUS_PATH = "/content/drive/MyDrive/InformationRetrieval/Data/clean_corpus.jsonl"
QUERIES_PATH = "/content/drive/MyDrive/InformationRetrieval/Data/clean_queries.jsonl"
QRELS_PATH = "/content/drive/MyDrive/InformationRetrieval/Data/qrels/test.tsv"

BIOWORDVEC_URL = "https://ftp.ncbi.nlm.nih.gov/pub/lu/Suppl/BioSentVec/BioWordVec_PubMed_MIMICIII_d200.vec.bin"
MODEL_PATH = "/content/BioWordVec_PubMed_MIMICIII_d200.vec.bin"
BIOWORDVEC_LIMIT = 2000000

OUTPUT_DIR_TFIDF = "/content/drive/MyDrive/InformationRetrieval/index_tfidf/"
OUTPUT_DIR_BIO = "/content/drive/MyDrive/InformationRetrieval/index_biowordvec/"

In [4]:
# ==================== HÀM CHUNG ====================
def load_jsonl(path: str) -> dict:
    data = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                data[obj["_id"]] = obj["text"]
    return data

def load_qrels(path: str) -> dict:
    qrels = defaultdict(dict)
    with open(path, encoding="utf-8") as f:
        next(f, None)
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 3: continue
            q_id, c_id, score = parts[0], parts[1], int(parts[2])
            qrels[q_id][c_id] = score
    return dict(qrels)

def download_biowordvec():
    if os.path.exists(MODEL_PATH):
        print(f"BioWordVec đã tồn tại: {MODEL_PATH}")
        return
    print("Đang tải BioWordVec (~13GB)...")
    subprocess.run(["wget", "-O", MODEL_PATH, BIOWORDVEC_URL], check=True)
    print("Tải BioWordVec hoàn tất!")

In [5]:
download_biowordvec()

Đang tải BioWordVec (~13GB)...
Tải BioWordVec hoàn tất!


In [6]:
# ==================== BUILD TF-IDF ====================
def build_tfidf_version(corpus):
    print("\n=== XÂY DỰNG TF-IDF THUẦN ===")
    doc_ids = list(corpus.keys())
    texts = list(corpus.values())

    tfidf = TfidfVectorizer(
        analyzer="word",
        token_pattern=r"(?u)\S+",
        min_df=1,
        max_df=0.9,
        sublinear_tf=True,
        norm='l2',
        use_idf=True,
        smooth_idf=False,
        ngram_range=(1, 2)
    )

    D_tfidf = tfidf.fit_transform(texts)
    print(f"TF-IDF Vocab size: {len(tfidf.vocabulary_):,} terms")

    os.makedirs(OUTPUT_DIR_TFIDF, exist_ok=True)
    save_npz(os.path.join(OUTPUT_DIR_TFIDF, "doc_matrix.npz"), D_tfidf)
    with open(os.path.join(OUTPUT_DIR_TFIDF, "doc_ids.pkl"), "wb") as f:
        pickle.dump(doc_ids, f)
    with open(os.path.join(OUTPUT_DIR_TFIDF, "tfidf_vectorizer.pkl"), "wb") as f:
        pickle.dump(tfidf, f)

    print(f"Index TF-IDF đã lưu tại: {OUTPUT_DIR_TFIDF}")
    return D_tfidf, doc_ids, tfidf

In [7]:
# ==================== PHIÊN BẢN 2: TF-IDF + BIOWORDVEC ====================
def get_word_vector(token: str, model):
    t = token.lower()
    if t in model:
        return model[t]
    if "_" in t:
        parts = [p for p in t.split("_") if p]
        vecs = [model[p] for p in parts if p in model]
        if vecs:
            return np.mean(vecs, axis=0).astype(np.float32)
    return None

def build_biowordvec_pure(corpus, model, dim, output_dir):
    print("\n=== XÂY DỰNG BIOWORDVEC THUẦN ===") # Dùng thêm tf để gán trọng số cho từ trong câu
    doc_ids = list(corpus.keys())
    N = len(doc_ids)
    D = np.zeros((N, dim), dtype=np.float32)

    print(f"Đang tính vector cho {N:,} documents...")

    for i, doc_id in enumerate(doc_ids):
        text = corpus[doc_id]
        tokens = text.split()
        if not tokens:
            continue

        tf_raw = defaultdict(int)
        for tok in tokens:
            tf_raw[tok.lower()] += 1
        total = len(tokens)

        weighted_sum = np.zeros(dim, dtype=np.float32)
        weight_sum = 0.0

        for tok, count in tf_raw.items():
            vec = get_word_vector(tok, model)
            if vec is None:
                continue
            tf = count / total
            weighted_sum += tf * vec
            weight_sum += tf

        if weight_sum > 1e-10:
            D[i] = weighted_sum / weight_sum

    # L2 Normalization
    norms = norm(D, axis=1, keepdims=True)
    norms = np.where(norms < 1e-10, 1.0, norms)
    D_norm = (D / norms).astype(np.float32)

    os.makedirs(output_dir, exist_ok=True)
    np.save(os.path.join(output_dir, "doc_matrix.npy"), D_norm)
    with open(os.path.join(output_dir, "doc_ids.pkl"), "wb") as f:
        pickle.dump(doc_ids, f)

    print(f"BioWordVec Pure index đã lưu tại: {output_dir}")
    return D_norm, doc_ids

In [8]:
# ==================== COMPUTE QUERY VECTORS ====================
def get_tfidf_query_vector(query_text, tfidf_vectorizer):
    q_vec = tfidf_vectorizer.transform([query_text])
    q_vec = normalize(q_vec, norm='l2', axis=1)
    return q_vec

def get_biowordvec_query_vector(query_text, model, dim):
    tokens = query_text.split()
    if not tokens:
        return np.zeros(dim, dtype=np.float32)

    tf_raw = defaultdict(int)
    for tok in tokens:
        tf_raw[tok.lower()] += 1
    total = len(tokens)

    weighted_sum = np.zeros(dim, dtype=np.float32)
    weight_sum = 0.0

    for tok, count in tf_raw.items():
        vec = get_word_vector(tok, model)
        if vec is None:
            continue
        tf = count / total
        weighted_sum += tf * vec
        weight_sum += tf

    if weight_sum > 1e-10:
        vec = weighted_sum / weight_sum
        return (vec / (norm(vec) + 1e-10)).astype(np.float32)
    return np.zeros(dim, dtype=np.float32)

In [9]:
# ==================== HYBRID LATE FUSION (Dot Product) ====================
def hybrid_search(query_text, tfidf_vectorizer, D_tfidf, D_bio, model, dim, lambda_weight=0.7):
    # TF-IDF score
    q_tfidf = get_tfidf_query_vector(query_text, tfidf_vectorizer)
    tfidf_scores = (D_tfidf @ q_tfidf.T).toarray().flatten()

    # BioWordVec score
    q_bio = get_biowordvec_query_vector(query_text, model, dim)
    bio_scores = D_bio @ q_bio

    # Late Fusion
    final_scores = lambda_weight * tfidf_scores + (1 - lambda_weight) * bio_scores
    return final_scores, tfidf_scores, bio_scores

In [10]:
# ==================== EVALUATION ====================
def evaluate_model(queries, qrels_dict, doc_ids,
                   tfidf_vectorizer=None, D_tfidf=None,
                   D_bio=None, model=None, dim=None,
                   lambda_weight=0.7, k=10, name="Model"):

    print(f"\nĐang đánh giá {name}...")
    run_dict = {}
    eval_queries = {q: t for q, t in queries.items() if q in qrels_dict}

    for i, (q_id, q_text) in enumerate(eval_queries.items(), 1):
        if name == "TF-IDF Thuần":
            q_vec = get_tfidf_query_vector(q_text, tfidf_vectorizer)
            scores = (D_tfidf @ q_vec.T).toarray().flatten()

        elif name == "BioWordVec Thuần":
            q_vec = get_biowordvec_query_vector(q_text, model, dim)
            scores = D_bio @ q_vec

        else:  # Hybrid Late Fusion
            scores, _, _ = hybrid_search(query_text=q_text,
                                         tfidf_vectorizer=tfidf_vectorizer,
                                         D_tfidf=D_tfidf,
                                         D_bio=D_bio,
                                         model=model,
                                         dim=dim,
                                         lambda_weight=lambda_weight)

        top_k_idx = np.argsort(scores)[::-1][:k]
        run_dict[q_id] = {doc_ids[idx]: float(scores[idx]) for idx in top_k_idx}

        if i % 100 == 0 or i == len(eval_queries):
            print(f" → Đã xử lý {i:4d}/{len(eval_queries)} queries")

    # Đánh giá
    evaluator = pytrec_eval.RelevanceEvaluator(
        qrels_dict,
        {
            "ndcg_cut_10",
            "recall_10",
            "recip_rank"
        }
    )
    results = evaluator.evaluate(run_dict)

    ndcg_scores = [v["ndcg_cut_10"] for v in results.values()]
    recall_scores = [v["recall_10"] for v in results.values()]
    mrr_scores = [v["recip_rank"] for v in results.values()]


    avg_ndcg = sum(ndcg_scores) / len(ndcg_scores)
    avg_recall = np.mean(recall_scores)
    avg_mrr = np.mean(mrr_scores)

    print(f"\n{'='*70}")
    print(f"KẾT QUẢ {name.upper()}")
    print(f"nDCG@10 : {avg_ndcg:.4f}")
    print(f"MRR@10    : {avg_mrr:.4f}")
    print(f"Recall@10 : {avg_recall:.4f}")
    print(f"{'='*70}")

    return avg_ndcg, avg_recall, avg_mrr

In [11]:
# ==================== CHẠY CHƯƠNG TRÌNH ====================
print("======= Load dữ liệu =======")
corpus = load_jsonl(CORPUS_PATH)
queries = load_jsonl(QUERIES_PATH)
qrels_dict = load_qrels(QRELS_PATH)

print(f"Corpus : {len(corpus):,} documents")
print(f"Queries: {len(queries):,} queries")
print(f"Qrels  : {len(qrels_dict):,} queries\n")

# 1. Build / Load TF-IDF
D_tfidf, doc_ids, tfidf_vectorizer = build_tfidf_version(corpus)

# 2. Load BioWordVec model
print("\nĐang load BioWordVec model...")
model = KeyedVectors.load_word2vec_format(
    MODEL_PATH, binary=True, limit=BIOWORDVEC_LIMIT, unicode_errors='ignore'
)
dim = model.vector_size
print(f"Loaded BioWordVec: {len(model):,} words, dim={dim}")

# 3. Build BioWordVec Thuần
D_bio, doc_ids_bio = build_biowordvec_pure(corpus, model, dim, OUTPUT_DIR_BIO)

assert doc_ids == doc_ids_bio, "Lỗi: doc_ids của hai index không khớp!"

======= Load dữ liệu =======
Corpus : 3,633 documents
Queries: 3,237 queries
Qrels  : 323 queries


=== XÂY DỰNG TF-IDF THUẦN ===
TF-IDF Vocab size: 331,615 terms
Index TF-IDF đã lưu tại: /content/drive/MyDrive/InformationRetrieval/index_tfidf/

Đang load BioWordVec model...
Loaded BioWordVec: 2,000,000 words, dim=200

=== XÂY DỰNG BIOWORDVEC THUẦN ===
Đang tính vector cho 3,633 documents...
BioWordVec Pure index đã lưu tại: /content/drive/MyDrive/InformationRetrieval/index_biowordvec/


In [12]:
# ====================== ĐÁNH GIÁ 3 MÔ HÌNH ======================
print("\n" + "="*85)
print("          BẮT ĐẦU ĐÁNH GIÁ SO SÁNH")
print("="*85)

results = {}

results["TF-IDF Thuần"] = evaluate_model(
    queries, qrels_dict, doc_ids,
    tfidf_vectorizer=tfidf_vectorizer, D_tfidf=D_tfidf,
    name="TF-IDF Thuần"
)

results["BioWordVec Thuần"] = evaluate_model(
    queries, qrels_dict, doc_ids,
    D_bio=D_bio, model=model, dim=dim,
    name="BioWordVec Thuần"
)

# Hybrid với nhiều lambda
lambdas = np.arange(0.5, 0.95, 0.05)
for lam in lambdas:
    name = f"Hybrid λ={lam}"
    ndcg, recall, mrr = evaluate_model(
        queries, qrels_dict, doc_ids,
        tfidf_vectorizer=tfidf_vectorizer, D_tfidf=D_tfidf,
        D_bio=D_bio, model=model, dim=dim,
        lambda_weight=lam,
        name=name
    )
    results[name] = (ndcg, recall, mrr)


          BẮT ĐẦU ĐÁNH GIÁ SO SÁNH

Đang đánh giá TF-IDF Thuần...
 → Đã xử lý  100/323 queries
 → Đã xử lý  200/323 queries
 → Đã xử lý  300/323 queries
 → Đã xử lý  323/323 queries

KẾT QUẢ TF-IDF THUẦN
nDCG@10 : 0.2511
MRR@10    : 0.4442
Recall@10 : 0.1091

Đang đánh giá BioWordVec Thuần...
 → Đã xử lý  100/323 queries
 → Đã xử lý  200/323 queries
 → Đã xử lý  300/323 queries
 → Đã xử lý  323/323 queries

KẾT QUẢ BIOWORDVEC THUẦN
nDCG@10 : 0.1695
MRR@10    : 0.3055
Recall@10 : 0.0760

Đang đánh giá Hybrid λ=0.5...
 → Đã xử lý  100/323 queries
 → Đã xử lý  200/323 queries
 → Đã xử lý  300/323 queries
 → Đã xử lý  323/323 queries

KẾT QUẢ HYBRID Λ=0.5
nDCG@10 : 0.2654
MRR@10    : 0.4660
Recall@10 : 0.1168

Đang đánh giá Hybrid λ=0.55...
 → Đã xử lý  100/323 queries
 → Đã xử lý  200/323 queries
 → Đã xử lý  300/323 queries
 → Đã xử lý  323/323 queries

KẾT QUẢ HYBRID Λ=0.55
nDCG@10 : 0.2718
MRR@10    : 0.4730
Recall@10 : 0.1198

Đang đánh giá Hybrid λ=0.6000000000000001...
 → Đã xử lý 

In [13]:
# ====================== BẢNG SO SÁNH ======================
print("\n" + "="*90)
print("                  BẢNG SO SÁNH KẾT QUẢ CUỐI CÙNG")
print("="*90)
print(f"{'Model':<35} {'nDCG@10':<10} {'Recall@10':<10} {'MRR@10':<10}")
print("-" * 70)
for model_name, (ndcg, recall, mrr) in results.items():
    print(f"{model_name:<35} {ndcg:<10.4f} {recall:<10.4f} {mrr:<10.4f}")
print("="*90)


                  BẢNG SO SÁNH KẾT QUẢ CUỐI CÙNG
Model                               nDCG@10    Recall@10  MRR@10    
----------------------------------------------------------------------
TF-IDF Thuần                        0.2511     0.1091     0.4442    
BioWordVec Thuần                    0.1695     0.0760     0.3055    
Hybrid λ=0.5                        0.2654     0.1168     0.4660    
Hybrid λ=0.55                       0.2718     0.1198     0.4730    
Hybrid λ=0.6000000000000001         0.2737     0.1218     0.4751    
Hybrid λ=0.6500000000000001         0.2757     0.1222     0.4837    
Hybrid λ=0.7000000000000002         0.2766     0.1232     0.4859    
Hybrid λ=0.7500000000000002         0.2760     0.1230     0.4829    
Hybrid λ=0.8000000000000003         0.2774     0.1234     0.4842    
Hybrid λ=0.8500000000000003         0.2756     0.1222     0.4844    
Hybrid λ=0.9000000000000004         0.2725     0.1221     0.4773    
